# Flash-MoE Windows + OpenVINO (Intel Arc / UMA Optimization)

Store a memoria limitata e runtime per esperti MoE con accelerazione **OpenVINO** per GPU Intel Arc (Lunar Lake / UMA).

### Obiettivi del Progetto:
- Leggere gli esperti da file separati sul disco con manifest strutturato;
- Mantenere una memoria RAM/VRAM contenuta (cache LRU a capacità limitata);
- Eseguire pre-fetching asincrono in background per nascondere la latenza I/O del disco;
- **Backend OpenVINO**: Eseguire il forward SwiGLU degli esperti tramite OpenVINO runtime (GPU Intel Arc / NPU) sfruttando la memoria condivisa (Zero-Copy);
- Router Top-K con Softmax per la selezione e combinazione pesata degli esperti.

In [1]:
from __future__ import annotations

from collections import OrderedDict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterator


@dataclass(frozen=True)
class ExpertKey:
    layer: int
    expert: int


@dataclass
class StoreStats:
    hits: int = 0
    misses: int = 0
    bytes_read: int = 0
    tensor_reads: int = 0


class ExpertStore:
    """Bounded expert store; inference backends consume the returned bytes."""

    def __init__(self, root: str | Path, capacity: int = 1, manifest: Any = None) -> None:
        if capacity < 1:
            raise ValueError("capacity must be at least 1")
        self.root = Path(root)
        self.capacity = capacity
        self.manifest = manifest
        self._cache: OrderedDict[ExpertKey, bytes] = OrderedDict()
        self.stats = StoreStats()

    def _path_for(self, key: ExpertKey) -> Path:
        if self.manifest is not None:
            return self.manifest.path_for(key)
        return self.root / f"layer_{key.layer:03d}" / f"expert_{key.expert:03d}.bin"

    def get(self, key: ExpertKey) -> bytes:
        cached = self._cache.get(key)
        if cached is not None:
            self._cache.move_to_end(key)
            self.stats.hits += 1
            return cached

        path = self._path_for(key)
        payload = path.read_bytes()
        self.stats.misses += 1
        self.stats.bytes_read += len(payload)
        self._cache[key] = payload
        self._cache.move_to_end(key)

        while len(self._cache) > self.capacity:
            self._cache.popitem(last=False)
        return payload

    def read_tensor(self, key: ExpertKey, name: str) -> bytes:
        """Read a tensor slice using in-memory cache if resident, or direct disk stream."""
        if self.manifest is None:
            raise ValueError("read_tensor requires a manifest with tensor slices")
        tensor = self.manifest.tensor_slice(key, name)
        
        if key in self._cache:
            self.stats.tensor_reads += 1
            return self._cache[key][tensor.offset : tensor.offset + tensor.nbytes]

        with self._path_for(key).open("rb") as handle:
            handle.seek(tensor.offset)
            payload = handle.read(tensor.nbytes)
        if len(payload) != tensor.nbytes:
            raise IOError(f"short read for {key}/{name}: expected {tensor.nbytes} bytes")
        self.stats.tensor_reads += 1
        self.stats.bytes_read += len(payload)
        return payload

    def clear(self) -> None:
        self._cache.clear()

    def resident_keys(self) -> tuple[ExpertKey, ...]:
        return tuple(self._cache)

    def route_layer(self, layer: int, expert_ids: Iterator[int]) -> Iterator[tuple[ExpertKey, bytes]]:
        for expert_id in expert_ids:
            key = ExpertKey(layer, expert_id)
            yield key, self.get(key)


In [2]:
import json
from dataclasses import dataclass


@dataclass(frozen=True)
class ExpertTensorSlice:
    name: str
    offset: int
    nbytes: int


@dataclass(frozen=True)
class ExpertRecord:
    key: ExpertKey
    relative_path: str
    nbytes: int
    tensors: dict[str, ExpertTensorSlice]


class ExpertManifest:
    def __init__(self, root: str | Path, records: dict[ExpertKey, ExpertRecord]) -> None:
        self.root = Path(root).resolve()
        self.records = records

    @classmethod
    def load(cls, manifest_path: str | Path) -> ExpertManifest:
        manifest_file = Path(manifest_path).resolve()
        payload = json.loads(manifest_file.read_text(encoding="utf-8"))
        records: dict[ExpertKey, ExpertRecord] = {}

        for item in payload.get("experts", []):
            key = ExpertKey(layer=int(item["layer"]), expert=int(item["expert"]))
            relative_path = str(item["path"])
            candidate = (manifest_file.parent / relative_path).resolve()
            if manifest_file.parent not in candidate.parents:
                raise ValueError(f"expert path escapes model directory: {relative_path}")
            if key in records:
                raise ValueError(f"duplicate expert: {key}")

            tensors: dict[str, ExpertTensorSlice] = {}
            intervals: list[tuple[int, int]] = []
            for name, tensor in item.get("tensors", {}).items():
                offset = int(tensor["offset"])
                nbytes = int(tensor["nbytes"])
                if offset < 0 or nbytes < 0:
                    raise ValueError(f"invalid tensor range for {key}: {name}")
                end = offset + nbytes
                if any(offset < other_end and other_offset < end for other_offset, other_end in intervals):
                    raise ValueError(f"overlapping tensor ranges for {key}")
                intervals.append((offset, end))
                tensors[name] = ExpertTensorSlice(name, offset, nbytes)

            total_nbytes = int(item["nbytes"])
            if any(end > total_nbytes for _, end in intervals):
                raise ValueError(f"tensor range exceeds expert size for {key}")
            records[key] = ExpertRecord(key, relative_path, total_nbytes, tensors)

        return cls(manifest_file.parent, records)

    def path_for(self, key: ExpertKey) -> Path:
        try:
            record = self.records[key]
        except KeyError as error:
            raise KeyError(f"expert not present in manifest: {key}") from error
        return self.root / record.relative_path

    def tensor_slice(self, key: ExpertKey, name: str) -> ExpertTensorSlice:
        try:
            return self.records[key].tensors[name]
        except KeyError as error:
            raise KeyError(f"tensor not present for expert {key}: {name}") from error


In [3]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as temporary_directory:
    root = Path(temporary_directory)
    (root / "layer_000").mkdir()
    (root / "layer_000" / "expert_000.bin").write_bytes(b"abc")
    (root / "manifest.json").write_text(
        json.dumps({
            "version": 1,
            "experts": [{"layer": 0, "expert": 0, "path": "layer_000/expert_000.bin", "nbytes": 3}],
        }),
        encoding="utf-8",
    )

    manifest = ExpertManifest.load(root / "manifest.json")
    assert manifest.path_for(ExpertKey(0, 0)).read_bytes() == b"abc"
    assert manifest.records[ExpertKey(0, 0)].nbytes == 3

print("ExpertManifest & ExpertStore base test: OK")

ExpertManifest & ExpertStore base test: OK


In [4]:
import struct

def unpack_f32(payload: bytes) -> list[float]:
    if len(payload) % 4 != 0:
        raise ValueError("f32 payload size must be divisible by 4")
    return list(struct.unpack(f"<{len(payload) // 4}f", payload))

def pack_f32(values: list[float]) -> bytes:
    return struct.pack(f"<{len(values)}f", *values)

def pack_expert(
    root: str | Path,
    key: ExpertKey,
    tensors: dict[str, bytes],
) -> dict[str, object]:
    required = ("gate", "up", "down")
    missing = [name for name in required if name not in tensors]
    if missing:
        raise ValueError(f"missing expert tensors: {', '.join(missing)}")

    root_path = Path(root)
    relative_path = Path("experts") / f"layer_{key.layer:03d}" / f"expert_{key.expert:03d}.bin"
    output_path = root_path / relative_path
    output_path.parent.mkdir(parents=True, exist_ok=True)

    offset = 0
    tensor_manifest: dict[str, dict[str, int]] = {}
    with output_path.open("wb") as output:
        for name in required:
            payload = tensors[name]
            tensor_manifest[name] = {"offset": offset, "nbytes": len(payload)}
            output.write(payload)
            offset += len(payload)

    return {
        "layer": key.layer,
        "expert": key.expert,
        "path": relative_path.as_posix(),
        "nbytes": offset,
        "tensors": tensor_manifest,
    }

def write_manifest(root: str | Path, records: list[dict[str, object]]) -> Path:
    manifest_path = Path(root) / "manifest.json"
    manifest_path.write_text(
        json.dumps({"version": 1, "experts": records}, indent=2),
        encoding="utf-8",
    )
    return manifest_path

## Probing Hardware OpenVINO

Verifichiamo la disponibilità dei dispositivi Intel (GPU Arc, NPU, CPU) rilevati da OpenVINO.

In [5]:
import openvino as ov

core = ov.Core()
devices = core.available_devices
print(f"OpenVINO Version: {ov.__version__}")
print(f"Dispositivi disponibili: {devices}")

for device in devices:
    full_name = core.get_property(device, "FULL_DEVICE_NAME")
    print(f" - {device}: {full_name}")

OpenVINO Version: 2026.4.0-22959-99c81491cc3-releases/2026/4
Dispositivi disponibili: ['CPU', 'GPU', 'NPU']
 - CPU: Intel(R) Core(TM) Ultra 5 226V
 - GPU: Intel(R) Arc(TM) 130V GPU (8GB) (iGPU)
 - NPU: Intel(R) AI Boost


## Backend OpenVINO (Zero-Copy & Hardware Accelerated)

Costruiamo la classe `OpenVINOExpertBackend` che compila un grafo OpenVINO per il calcolo SwiGLU.
Il grafo riceve l'attivazione `x` e le matrici degli pesi `gate`, `up`, `down` come input `ov.Tensor` estratti dallo `ExpertStore` senza fare copie ridondanti.

In [6]:
import openvino as ov
import openvino.opset13 as ops
import numpy as np

class OpenVINOExpertBackend:
    """OpenVINO-accelerated SwiGLU expert execution backend."""

    def __init__(self, store: ExpertStore, device: str = "AUTO"):
        self.store = store
        self.core = ov.Core()
        available = self.core.available_devices
        if device == "GPU" and "GPU" in available:
            self.target_device = "GPU"
        elif "GPU" in available:
            self.target_device = "GPU"
        else:
            self.target_device = "CPU"
        
        self._compiled_models: dict[tuple[int, int], ov.CompiledModel] = {}
        print(f"[OpenVINOBackend] Target device: {self.target_device}")

    def _get_compiled_model(self, hidden_size: int, intermediate_size: int) -> ov.CompiledModel:
        shape_key = (hidden_size, intermediate_size)
        if shape_key in self._compiled_models:
            return self._compiled_models[shape_key]

        x = ops.parameter([1, hidden_size], ov.Type.f32, name="activation")
        gate = ops.parameter([intermediate_size, hidden_size], ov.Type.f32, name="gate")
        up = ops.parameter([intermediate_size, hidden_size], ov.Type.f32, name="up")
        down = ops.parameter([hidden_size, intermediate_size], ov.Type.f32, name="down")

        gate_proj = ops.matmul(x, gate, transpose_a=False, transpose_b=True)
        up_proj = ops.matmul(x, up, transpose_a=False, transpose_b=True)
        swish_gate = ops.swish(gate_proj)
        swiglu_out = ops.multiply(swish_gate, up_proj)
        out = ops.matmul(swiglu_out, down, transpose_a=False, transpose_b=True)

        model = ov.Model([out], [x, gate, up, down], name="SwiGLU_Expert")
        compiled = self.core.compile_model(model, self.target_device)
        self._compiled_models[shape_key] = compiled
        return compiled

    def forward(
        self,
        key: ExpertKey,
        activation: list[float],
        hidden_size: int,
        intermediate_size: int,
    ) -> list[float]:
        gate_bytes = self.store.read_tensor(key, "gate")
        up_bytes = self.store.read_tensor(key, "up")
        down_bytes = self.store.read_tensor(key, "down")

        gate_arr = np.frombuffer(gate_bytes, dtype=np.float32).reshape(intermediate_size, hidden_size)
        up_arr = np.frombuffer(up_bytes, dtype=np.float32).reshape(intermediate_size, hidden_size)
        down_arr = np.frombuffer(down_bytes, dtype=np.float32).reshape(hidden_size, intermediate_size)
        x_arr = np.array(activation, dtype=np.float32).reshape(1, hidden_size)

        compiled = self._get_compiled_model(hidden_size, intermediate_size)
        infer_request = compiled.create_infer_request()
        
        results = infer_request.infer({
            "activation": x_arr,
            "gate": gate_arr,
            "up": up_arr,
            "down": down_arr,
        })

        output_tensor = list(results.values())[0]
        return output_tensor.flatten().tolist()


In [7]:
import math

class CPUExpertBackend:
    """Reference CPU backend for SwiGLU cross-validation."""

    def __init__(self, store: ExpertStore):
        self.store = store

    @staticmethod
    def _matvec(matrix: list[float], vector: list[float], rows: int, columns: int) -> list[float]:
        return [
            sum(matrix[row * columns + column] * vector[column] for column in range(columns))
            for row in range(rows)
        ]

    def forward(
        self,
        key: ExpertKey,
        activation: list[float],
        hidden_size: int,
        intermediate_size: int,
    ) -> list[float]:
        gate = unpack_f32(self.store.read_tensor(key, "gate"))
        up = unpack_f32(self.store.read_tensor(key, "up"))
        down = unpack_f32(self.store.read_tensor(key, "down"))

        gate_values = self._matvec(gate, activation, intermediate_size, hidden_size)
        up_values = self._matvec(up, activation, intermediate_size, hidden_size)
        mixed = [
            (v / (1.0 + math.exp(-v))) * u_v
            for v, u_v in zip(gate_values, up_values)
        ]
        return self._matvec(down, mixed, hidden_size, intermediate_size)


In [8]:
with TemporaryDirectory() as temporary_directory:
    root = Path(temporary_directory)
    tensors = {
        "gate": pack_f32([1.0, 0.5, -0.5, 1.5]),
        "up": pack_f32([0.8, -0.2, 0.1, 1.1]),
        "down": pack_f32([0.5, 0.2, -0.1, 0.9]),
    }
    record = pack_expert(root, ExpertKey(0, 0), tensors)
    manifest = ExpertManifest.load(write_manifest(root, [record]))
    store = ExpertStore(root, capacity=1, manifest=manifest)

    cpu_backend = CPUExpertBackend(store)
    ov_backend = OpenVINOExpertBackend(store)

    key = ExpertKey(0, 0)
    activation = [1.2, -0.8]
    cpu_res = cpu_backend.forward(key, activation, 2, 2)
    ov_res = ov_backend.forward(key, activation, 2, 2)

    print(f"Risultato CPU:      {cpu_res}")
    print(f"Risultato OpenVINO: {ov_res}")
    assert all(math.isclose(c, o, abs_tol=1e-3) for c, o in zip(cpu_res, ov_res))
    print("Validazione CPU vs OpenVINO: OK!")


[OpenVINOBackend] Target device: GPU
Risultato CPU:      [0.34791902496904137, 0.11282531509960014]
Risultato OpenVINO: [0.34808623790740967, 0.11259382963180542]
Validazione CPU vs OpenVINO: OK!


## Router Top-K con Softmax per Selezione Esperti

Il router proietta l'attivazione di input sugli esperti disponibili, calcola Softmax sui `top_k` valori più alti e restituisce sia gli ID degli esperti sia i rispettivi pesi di combinazione.

In [9]:
class ExpertRouter:
    """Gating network that selects top-k experts and returns normalized softmax weights."""
    def __init__(self, hidden_size: int, num_experts: int, top_k: int = 2):
        self.hidden_size = hidden_size
        self.num_experts = num_experts
        self.top_k = min(top_k, num_experts)
        np.random.seed(42)
        self.gate_weights = np.random.randn(hidden_size, num_experts).astype(np.float32)

    def route(self, activation: list[float]) -> tuple[list[int], list[float]]:
        scores = np.dot(activation, self.gate_weights)
        top_k_indices = np.argsort(scores)[-self.top_k:][::-1].tolist()
        top_k_scores = scores[top_k_indices]
        
        exp_scores = np.exp(top_k_scores - np.max(top_k_scores))
        weights = (exp_scores / np.sum(exp_scores)).tolist()
        return top_k_indices, weights

router = ExpertRouter(hidden_size=2, num_experts=4, top_k=2)
selected_ids, weights = router.route([1.0, -1.0])
print(f"Esperti selezionati: {selected_ids}")
print(f"Pesi Softmax:        {weights}")


Esperti selezionati: [3, 0]
Pesi Softmax:        [0.5061815770323428, 0.49381842296765716]


## Pre-fetching Asincrono e Runtime MoE Concorrente

`AsyncExpertStore` usa un `ThreadPoolExecutor` in background. Quando il runtime processa una richiesta al passo $N$, avvia il pre-caricamento dell'esperto $N+1$. Quando la GPU/OpenVINO richiede i tensori dell'esperto, questo è già residente nella cache RAM.

In [10]:
from concurrent.futures import ThreadPoolExecutor, Future

class RAMBoundedExpertStore(ExpertStore):
    def __init__(self, root: str | Path, capacity: int = 2, manifest: Any = None) -> None:
        if capacity > 2:
            raise ValueError("Capacity cannot exceed 2 for bounded RAM policy")
        super().__init__(root=root, capacity=capacity, manifest=manifest)


class AsyncExpertStore(RAMBoundedExpertStore):
    def __init__(self, root: str | Path, capacity: int = 2, manifest: Any = None):
        super().__init__(root, capacity, manifest)
        self.executor = ThreadPoolExecutor(max_workers=1)
        self.pending_loads: dict[ExpertKey, Future[bytes]] = {}

    def prefetch(self, key: ExpertKey) -> None:
        if key in self._cache or key in self.pending_loads:
            return

        def _load():
            path = self._path_for(key)
            return path.read_bytes()

        self.pending_loads[key] = self.executor.submit(_load)

    def get(self, key: ExpertKey) -> bytes:
        future = self.pending_loads.pop(key, None)
        if future is not None:
            payload = future.result()
            self.stats.misses += 1
            self.stats.bytes_read += len(payload)
            self._cache[key] = payload
            self._cache.move_to_end(key)
            while len(self._cache) > self.capacity:
                self._cache.popitem(last=False)
            return payload

        return super().get(key)

    def close(self) -> None:
        self.executor.shutdown(wait=True)


class MoERuntime:
    def __init__(self, store: AsyncExpertStore, backend: Any, router: ExpertRouter, intermediate_size: int | None = None):
        self.store = store
        self.backend = backend
        self.router = router
        self.intermediate_size = intermediate_size

    def process_token(self, layer: int, activation: list[float], next_expert_key: ExpertKey | None = None) -> list[float]:
        if next_expert_key is not None:
            self.store.prefetch(next_expert_key)

        expert_ids, weights = self.router.route(activation)
        hidden_dim = len(activation)
        inter_dim = self.intermediate_size if self.intermediate_size is not None else hidden_dim

        combined_output = np.zeros(hidden_dim, dtype=np.float32)
        for expert_id, weight in zip(expert_ids, weights):
            key = ExpertKey(layer, expert_id)
            self.store.get(key)
            expert_out = self.backend.forward(key, activation, hidden_dim, inter_dim)
            combined_output += weight * np.array(expert_out, dtype=np.float32)

        return combined_output.tolist()


## Benchmark End-to-End: OpenVINO + Pre-fetching Asincrono

Eseguiamo una simulazione batch di inferenza MoE su 20 token sequenziali usando il backend OpenVINO e misuriamo l'efficienza dello Store e del Prefetching.

In [11]:
import time

with TemporaryDirectory() as temporary_directory:
    root = Path(temporary_directory)
    records = []
    num_experts = 4
    hidden_dim = 64
    inter_dim = 128

    for eid in range(num_experts):
        tensors = {
            "gate": pack_f32(np.random.randn(inter_dim * hidden_dim).tolist()),
            "up": pack_f32(np.random.randn(inter_dim * hidden_dim).tolist()),
            "down": pack_f32(np.random.randn(hidden_dim * inter_dim).tolist()),
        }
        records.append(pack_expert(root, ExpertKey(0, eid), tensors))

    manifest = ExpertManifest.load(write_manifest(root, records))
    store = AsyncExpertStore(root, capacity=2, manifest=manifest)
    backend = OpenVINOExpertBackend(store)
    router = ExpertRouter(hidden_size=hidden_dim, num_experts=num_experts, top_k=2)
    runtime = MoERuntime(store, backend, router, intermediate_size=inter_dim)

    print("--- INIZIO ESECUZIONE BATCH MOE --- ")
    start_time = time.perf_counter()

    tokens = [np.random.randn(hidden_dim).tolist() for _ in range(20)]
    for i, token in enumerate(tokens):
        next_key = ExpertKey(0, (i + 1) % num_experts) if i + 1 < len(tokens) else None
        out = runtime.process_token(layer=0, activation=token, next_expert_key=next_key)

    total_time = time.perf_counter() - start_time
    print(f"Tempo totale esecuzione (20 token): {total_time:.4f} secondi")
    print(f"Latenza media per token:             {(total_time / 20) * 1000:.2f} ms")
    print(f"Cache Stats -> Hits: {store.stats.hits}, Misses: {store.stats.misses}, Bytes Letti: {store.stats.bytes_read} B")
    store.close()


[OpenVINOBackend] Target device: GPU
--- INIZIO ESECUZIONE BATCH MOE --- 
Tempo totale esecuzione (20 token): 0.1680 secondi
Latenza media per token:             8.40 ms
Cache Stats -> Hits: 12, Misses: 28, Bytes Letti: 2752512 B
